# Data Cleaning and Quality Validation

## Purpose

This notebook cleans, standardizes, and validates the 2025 Airline On-Time Performance dataset before feature engineering and machine learning.

The workflow preserves the original raw dataset while producing a reliable managed Delta table for downstream analytical tasks.

The data-cleaning and quality-validation process includes:

1. Loading the cleaned source configuration and managed raw Delta table
2. Validating the approved 32-column schema
3. Standardizing dates, binary indicators, numerical fields, and categorical text
4. Verifying date parsing and converted data types
5. Detecting exact and business-key duplicate records
6. Measuring and investigating missing values
7. Identifying structurally valid null values associated with cancellations and diversions
8. Removing incomplete records that cannot support supervised learning
9. Validating permitted categorical and binary value domains
10. Checking operational relationships between related variables
11. Confirming consistency between:
    - `DEP_DELAY` and `DEP_DEL15`
    - `ARR_DELAY` and `ARR_DEL15`
    - `CANCELLED` and `CANCELLATION_CODE`
    - `DIVERTED`, `CANCELLED`, and `ARR_DEL15`
12. Saving the validated dataset as the managed Delta table `workspace.default.flights_clean`

The resulting table serves as the input for the `05_feature_engineering` notebook.

## 1. Environment and Path Configuration

The raw dataset is read from the managed Unity Catalog table created during
data ingestion. The cleaned dataset will later be written in Delta format to
the `processed` directory of the project Unity Catalog Volume.

In [0]:
from __future__ import annotations

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T


# ---------------------------------------------------------------------
# Source and destination
# ---------------------------------------------------------------------

RAW_TABLE = "workspace.default.flights_raw"

PROCESSED_BASE_PATH = (
    "/Volumes/workspace/default/"
    "flight_delay_capstone/processed"
)

CLEAN_DELTA_PATH = f"{PROCESSED_BASE_PATH}/flights_clean"

TARGET_COLUMN = "ARR_DEL15"


print("Data-cleaning configuration loaded successfully.")
print(f"Source table: {RAW_TABLE}")
print(f"Processed layer: {PROCESSED_BASE_PATH}")
print(f"Clean Delta output: {CLEAN_DELTA_PATH}")
print(f"Prediction target: {TARGET_COLUMN}")

## 2. Load the Raw Dataset

The cleaning process begins by loading the `flights_raw` managed Delta table.
This table already combines the twelve monthly BTS files for January through
December 2025, so the source CSV files do not need to be read again.

In [0]:
def require_table(table_name: str) -> None:
    """Validate that a required Unity Catalog table exists."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-ingestion notebook before continuing."
        )


require_table(RAW_TABLE)

df_raw: DataFrame = spark.table(RAW_TABLE)

raw_row_count = df_raw.count()
raw_column_count = len(df_raw.columns)

print("Raw dataset loaded successfully.")
print(f"Total records: {raw_row_count:,}")
print(f"Total columns: {raw_column_count}")

In [0]:
display(df_raw.limit(10))

## 3. Schema Validation

The raw dataset is compared with the approved list of 32 BTS attributes.
This validation identifies missing, unexpected, or incorrectly named columns
before any cleaning rules are applied.

In [0]:
EXPECTED_COLUMNS = [
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_NM",
    "DEST",
    "DEST_CITY_NAME",
    "DEST_STATE_NM",
    "CRS_DEP_TIME",
    "DEP_DELAY",
    "DEP_DEL15",
    "TAXI_OUT",
    "TAXI_IN",
    "CRS_ARR_TIME",
    "ARR_DELAY",
    "ARR_DEL15",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
    "CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

actual_columns = df_raw.columns

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(actual_columns))
unexpected_columns = sorted(set(actual_columns) - set(EXPECTED_COLUMNS))

print(f"Expected columns: {len(EXPECTED_COLUMNS)}")
print(f"Actual columns: {len(actual_columns)}")

if missing_columns:
    raise ValueError(
        "Schema validation failed. Missing required columns: "
        f"{missing_columns}"
    )

print("All required columns are present.")

if unexpected_columns:
    print(f"Unexpected columns detected: {unexpected_columns}")
else:
    print("No unexpected columns detected.")

In [0]:
schema_rows = [
    (field.name, field.dataType.simpleString(), field.nullable)
    for field in df_raw.schema.fields
]

schema_df = spark.createDataFrame(
    schema_rows,
    ["COLUMN_NAME", "CURRENT_DATA_TYPE", "NULLABLE"],
)

display(schema_df)

## 4. Data Type and Text Standardization

Several fields were inferred using generic data types during ingestion. The following standardizations are applied:

- `FL_DATE` is converted from a string containing date and time into Spark `DateType`.
- Binary indicator fields are converted to integers.
- Code and location fields are trimmed and standardized.
- Empty categorical values are converted to null.
- Numerical performance variables remain numeric.

The raw dataset is not modified, and no records are removed during this stage.

In [0]:
BINARY_COLUMNS = [
    "DEP_DEL15",
    "ARR_DEL15",
    "CANCELLED",
    "DIVERTED",
]

CODE_COLUMNS = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "ORIGIN_STATE_NM",
    "DEST",
    "DEST_STATE_NM",
    "CANCELLATION_CODE",
]

TEXT_COLUMNS = [
    "ORIGIN_CITY_NAME",
    "DEST_CITY_NAME",
]

INTEGER_COLUMNS = [
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "OP_CARRIER_FL_NUM",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
]

DOUBLE_COLUMNS = [
    "DEP_DELAY",
    "TAXI_OUT",
    "TAXI_IN",
    "ARR_DELAY",
    "CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

In [0]:
df_clean = df_raw

# Convert FL_DATE from strings such as:
# "12/1/2025 12:00:00 AM"
# into Spark DateType.
df_clean = df_clean.withColumn(
    "FL_DATE",
    F.to_date(
        F.try_to_timestamp(
            F.trim(F.col("FL_DATE")),
            F.lit("M/d/yyyy h:mm:ss a"),
        )
    ),
)

# Convert binary indicators to integers.
for column_name in BINARY_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

# Standardize integer columns.
for column_name in INTEGER_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.IntegerType()),
    )

# Standardize numeric columns.
for column_name in DOUBLE_COLUMNS:
    df_clean = df_clean.withColumn(
        column_name,
        F.col(column_name).cast(T.DoubleType()),
    )

# Trim, uppercase, and convert blank code values to null.
for column_name in CODE_COLUMNS:
    cleaned_value = F.upper(F.trim(F.col(column_name)))

    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

# Trim readable text fields and convert blank values to null.
for column_name in TEXT_COLUMNS:
    cleaned_value = F.trim(F.col(column_name))

    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            cleaned_value.isNull() | (cleaned_value == ""),
            F.lit(None),
        ).otherwise(cleaned_value),
    )

print("Initial type and text standardization completed.")

## 5. Date Parsing Validation

The `FL_DATE` column was converted from a string to a Spark date during the data type standardization stage. This validation checks whether any original non-null date values failed to convert successfully.

The following measures are reviewed:

- **TOTAL_ROWS** — Total number of records in the raw dataset.
- **ORIGINAL_NULL_DATES** — Number of records where `FL_DATE` was already missing in the raw data.
- **DATE_PARSE_FAILURES** — Number of non-null date values that could not be converted to a valid Spark date.

A value of **0** for `DATE_PARSE_FAILURES` confirms that all available flight dates were parsed successfully.

In [0]:
date_validation = (
    df_raw
    .select(
        F.count("*").alias("TOTAL_ROWS"),
        F.sum(
            F.when(F.col("FL_DATE").isNull(), 1).otherwise(0)
        ).alias("ORIGINAL_NULL_DATES"),
        F.sum(
            F.when(
                F.col("FL_DATE").isNotNull()
                & F.try_to_timestamp(
                    F.trim(F.col("FL_DATE")),
                    F.lit("M/d/yyyy h:mm:ss a"),
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias("DATE_PARSE_FAILURES"),
    )
)

display(date_validation)

### Validation Interpretation

The date parsing results confirm whether the `FL_DATE` conversion was successful. If `DATE_PARSE_FAILURES` is equal to **0**, all non-null source dates were converted correctly and no date-related cleaning action is required. Any failed records must be inspected before continuing with duplicate and missing-value analysis.

## 6. Schema Verification

After applying the initial data type standardization, the schema is verified to confirm that the selected columns have been converted to their intended data types. This validation ensures that the dataset is ready for subsequent data cleaning, feature engineering, and machine learning tasks.

The verification focuses on the following columns:

- `FL_DATE` — should be stored as a Spark `date`.
- `DEP_DEL15` — should be stored as an integer binary indicator.
- `ARR_DEL15` — should be stored as an integer binary indicator.
- `CANCELLED` — should be stored as an integer binary indicator.
- `DIVERTED` — should be stored as an integer binary indicator.

The resulting schema is displayed below for verification.

In [0]:
converted_columns = [
    "FL_DATE",
    "DEP_DEL15",
    "ARR_DEL15",
    "CANCELLED",
    "DIVERTED",
]

converted_schema_rows = [
    (
        field.name,
        field.dataType.simpleString(),
        field.nullable,
    )
    for field in df_clean.select(*converted_columns).schema.fields
]

converted_schema_df = spark.createDataFrame(
    converted_schema_rows,
    ["COLUMN_NAME", "STANDARDIZED_DATA_TYPE", "NULLABLE"],
)

display(converted_schema_df)

## 7. Preview of Standardized Dataset

A sample of the standardized dataset is displayed below to verify that the applied transformations were successful. The preview confirms that the selected variables have been converted to their expected data types and standardized according to the project's data quality requirements.

The preview is used to visually verify the following:

- `FL_DATE` has been successfully converted to Spark's `DateType`.
- Binary indicator variables (`DEP_DEL15`, `ARR_DEL15`, `CANCELLED`, and `DIVERTED`) are stored as integer values.
- Airline and airport codes have been standardized by trimming unnecessary whitespace and converting text to uppercase.
- The selected variables remain consistent with the approved project schema before proceeding to the next data cleaning stage.

In [0]:
display(
    df_clean.select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "OP_CARRIER_FL_NUM",
        "ORIGIN",
        "DEST",
        "DEP_DEL15",
        "ARR_DEL15",
        "CANCELLED",
        "DIVERTED",
    ).limit(20)
)

## 8. Duplicate Detection

Duplicate records can distort descriptive statistics, bias model training, and produce inaccurate operational insights. Two types of duplicates are evaluated:

1. **Exact duplicates** — Records in which all 32 variables contain identical values.
2. **Business-key duplicates** — Records that share the same flight identity based on the flight date, airline, flight number, route, and scheduled departure time.

The business key used in this analysis consists of:

- `FL_DATE`
- `OP_UNIQUE_CARRIER`
- `OP_CARRIER_FL_NUM`
- `ORIGIN`
- `DEST`
- `CRS_DEP_TIME`

Duplicate records are inspected before removal because multiple records with similar identifiers may represent legitimate flight operations or data corrections.

In [0]:
BUSINESS_KEY_COLUMNS = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
]

print("Business key configured successfully.")
print("Business key columns:")
for column_name in BUSINESS_KEY_COLUMNS:
    print(f"- {column_name}")

### 8.1 Exact Duplicate Analysis

Exact duplicate analysis compares the total number of standardized records with the number of distinct records across all columns. The difference represents records that are completely identical.

No records are removed during this analysis.

In [0]:
# Cache the standardized DataFrame because it will be reused
# across several data-quality checks.
standardized_row_count = df_clean.count()

distinct_row_count = df_clean.dropDuplicates().count()

exact_duplicate_count = standardized_row_count - distinct_row_count

exact_duplicate_summary = spark.createDataFrame(
    [
        (
            standardized_row_count,
            distinct_row_count,
            exact_duplicate_count,
        )
    ],
    [
        "TOTAL_STANDARDIZED_ROWS",
        "DISTINCT_ROWS",
        "EXACT_DUPLICATE_ROWS",
    ],
)

display(exact_duplicate_summary)

### Validation Summary

The exact duplicate analysis indicates that the standardized dataset contains **700,619** unique records, with **no exact duplicate rows** detected across all selected variables. Since the number of distinct records is equal to the total number of standardized records, no duplicate removal is required at the full-record level.

The dataset therefore preserves all observations and is suitable for further duplicate validation using the business key defined for this project.

## 8.2 Business Key Duplicate Analysis

While no exact duplicate records were detected, multiple records may still represent the same scheduled flight. Therefore, a business-key duplicate analysis is performed using the flight identity defined by the project.

The business key consists of:

- `FL_DATE`
- `OP_UNIQUE_CARRIER`
- `OP_CARRIER_FL_NUM`
- `ORIGIN`
- `DEST`
- `CRS_DEP_TIME`

Records sharing the same business key are identified and reviewed before any cleaning action is taken. At this stage, no records are removed automatically.

In [0]:
BUSINESS_KEY_COLUMNS = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
]

business_key_duplicates = (
    df_clean
    .groupBy(*BUSINESS_KEY_COLUMNS)
    .agg(F.count("*").alias("RECORD_COUNT"))
    .filter(F.col("RECORD_COUNT") > 1)
    .orderBy(F.col("RECORD_COUNT").desc())
)

duplicate_business_key_count = business_key_duplicates.count()

duplicate_summary_row = (
    business_key_duplicates
    .agg(
        F.coalesce(
            F.sum("RECORD_COUNT"),
            F.lit(0),
        ).cast("long").alias("TOTAL_DUPLICATE_RECORDS"),
        F.coalesce(
            F.sum(F.col("RECORD_COUNT") - 1),
            F.lit(0),
        ).cast("long").alias("ADDITIONAL_RECORDS_BEYOND_FIRST"),
    )
    .first()
)

total_duplicate_records = int(
    duplicate_summary_row["TOTAL_DUPLICATE_RECORDS"]
)

additional_records_beyond_first = int(
    duplicate_summary_row["ADDITIONAL_RECORDS_BEYOND_FIRST"]
)

business_key_summary = spark.createDataFrame(
    [
        (
            int(duplicate_business_key_count),
            total_duplicate_records,
            additional_records_beyond_first,
        )
    ],
    schema="""
        DUPLICATE_BUSINESS_KEYS long,
        TOTAL_DUPLICATE_RECORDS long,
        ADDITIONAL_RECORDS_BEYOND_FIRST long
    """,
)

display(business_key_summary)

### Validation Summary

The business-key duplicate analysis found no repeated flight identities based on the selected combination of flight date, airline, flight number, origin, destination, and scheduled departure time. Therefore, no records need to be removed based on the defined business key.

Together with the exact duplicate analysis, this result confirms that the standardized dataset contains no duplicate records requiring corrective action.

## 9. Missing Value Analysis

Missing values are evaluated across all selected variables to determine whether they represent data-quality issues or expected operational conditions.

Some null values may be valid. For example:

- `CANCELLATION_CODE` is normally null when a flight is not cancelled.
- Delay-cause fields may be null when a flight is not delayed by at least 15 minutes.
- Actual flight-operation fields may be null for cancelled or diverted flights.

Therefore, missing values are measured before any treatment is applied. No records are removed or imputed during this stage.

In [0]:
total_rows = df_clean.count()

null_summary_expressions = []

for column_name in df_clean.columns:
    null_count_expression = F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(column_name)

    null_summary_expressions.append(null_count_expression)

null_counts_row = (
    df_clean
    .select(*null_summary_expressions)
    .first()
)

null_summary_rows = []

for column_name in df_clean.columns:
    null_count = int(null_counts_row[column_name])

    null_percentage = (
        (null_count / total_rows) * 100
        if total_rows > 0
        else 0.0
    )

    null_summary_rows.append(
        (
            column_name,
            null_count,
            round(null_percentage, 4),
        )
    )

null_summary_df = spark.createDataFrame(
    null_summary_rows,
    schema="""
        COLUMN_NAME string,
        NULL_COUNT long,
        NULL_PERCENTAGE double
    """,
)

null_summary_df = null_summary_df.orderBy(
    F.col("NULL_PERCENTAGE").desc(),
    F.col("COLUMN_NAME"),
)

display(null_summary_df)

### Initial Interpretation

The missing-value summary reports the number and percentage of null values for every selected variable.

High null percentages do not automatically indicate poor data quality. Operational context must be considered before applying any cleaning rule. In particular, cancellation indicators, delay-cause fields, and actual flight-performance variables may contain structurally valid null values.

The next analysis separates:

1. Columns with no missing values.
2. Columns with expected conditional missingness.
3. Columns that require further inspection.

In [0]:
columns_with_nulls_df = (
    null_summary_df
    .filter(F.col("NULL_COUNT") > 0)
)

display(columns_with_nulls_df)

### Columns Containing Missing Values

The table above isolates variables containing at least one null value. These columns will be reviewed based on their operational meaning and relationship with flight status.

No missing-value treatment will be applied until the conditional patterns associated with cancelled, diverted, delayed, and completed flights are examined.

In [0]:
missing_value_overview = (
    null_summary_df
    .agg(
        F.sum(
            F.when(F.col("NULL_COUNT") > 0, 1).otherwise(0)
        ).alias("COLUMNS_WITH_NULLS"),
        F.sum(
            F.when(F.col("NULL_COUNT") == 0, 1).otherwise(0)
        ).alias("COLUMNS_WITHOUT_NULLS"),
        F.max("NULL_PERCENTAGE").alias(
            "HIGHEST_NULL_PERCENTAGE"
        ),
    )
)

display(missing_value_overview)

### Validation Summary

The missing-value analysis establishes the baseline level of completeness across the standardized dataset. Columns containing null values will be evaluated using business rules before any imputation or record removal is performed.

This prevents valid operational nulls from being incorrectly treated as data errors.

## 10. Missing Value Investigation

The previous analysis identified several variables containing null values. Before applying any cleaning rule, it is necessary to determine whether these null values represent data quality issues or expected operational conditions.

The following investigation examines the relationship between missing values and the operational status of flights, particularly cancellations and diversions.

In [0]:
display(

    df_clean.groupBy(
        "CANCELLED",
        "DIVERTED"
    ).agg(

        F.count("*").alias("TOTAL"),

        F.sum(
            F.when(
                F.col("ARR_DEL15").isNull(),
                1
            ).otherwise(0)
        ).alias("NULL_TARGET")

    )

)

## 10.1 Investigation of the Remaining Missing Target Record

The previous analysis confirmed that nearly all missing values in `ARR_DEL15` are associated with cancelled or diverted flights, which is expected because these flights do not complete a normal arrival process.

However, one record remains where:

- `CANCELLED = 0`
- `DIVERTED = 0`
- `ARR_DEL15` is null

This record is reviewed separately to determine whether it represents incomplete reporting or another data-quality issue.

In [0]:
remaining_missing_target_record = (
    df_clean
    .filter(
        (F.col("CANCELLED") == 0)
        & (F.col("DIVERTED") == 0)
        & F.col("ARR_DEL15").isNull()
    )
)

print(
    "Remaining non-cancelled, non-diverted records "
    f"with missing ARR_DEL15: {remaining_missing_target_record.count()}"
)

display(remaining_missing_target_record)

### Investigation Summary

The remaining record with a missing `ARR_DEL15` value is inspected to determine whether the target can be derived reliably from `ARR_DELAY` or whether the record should be excluded due to incomplete operational reporting.

No cleaning action is applied until the values in the record are reviewed.

## 10.2 Cleaning Decision

The investigation identified one record where the target variable (`ARR_DEL15`) was missing despite the flight not being cancelled or diverted.

Because the target variable is required for supervised machine learning and cannot be reliably reconstructed from the available information, this record is considered incomplete and will be excluded from the cleaned dataset.

All remaining missing values are associated with legitimate operational conditions such as cancellations or diversions and will be retained.

In [0]:
invalid_target_records = (
    df_clean.filter(
        (F.col("CANCELLED") == 0)
        &
        (F.col("DIVERTED") == 0)
        &
        (F.col("ARR_DEL15").isNull())
    )
)

invalid_record_count = invalid_target_records.count()

print(f"Invalid target records: {invalid_record_count}")

df_clean = (
    df_clean.filter(
        ~(
            (F.col("CANCELLED") == 0)
            &
            (F.col("DIVERTED") == 0)
            &
            (F.col("ARR_DEL15").isNull())
        )
    )
)

print(f"Rows after cleaning: {df_clean.count():,}")

## 11. Domain Validation

Domain validation verifies that selected categorical and binary variables contain only values permitted by the BTS data dictionary and the project’s analytical definitions.

The following domains are evaluated:

- `QUARTER`: integers from 1 to 4
- `MONTH`: integers from 1 to 12
- `DAY_OF_WEEK`: integers from 1 to 7
- `DEP_DEL15`: binary values 0 or 1
- `ARR_DEL15`: binary values 0 or 1, or null for cancelled and diverted flights
- `CANCELLED`: binary values 0 or 1
- `DIVERTED`: binary values 0 or 1

Null values are evaluated separately and are not automatically considered invalid during this validation.

In [0]:
DOMAIN_RULES = {
    "QUARTER": [1, 2, 3, 4],
    "MONTH": list(range(1, 13)),
    "DAY_OF_WEEK": list(range(1, 8)),
    "DEP_DEL15": [0, 1],
    "ARR_DEL15": [0, 1],
    "CANCELLED": [0, 1],
    "DIVERTED": [0, 1],
}

domain_validation_rows = []

for column_name, valid_values in DOMAIN_RULES.items():
    invalid_count = (
        df_clean
        .filter(
            F.col(column_name).isNotNull()
            & ~F.col(column_name).isin(valid_values)
        )
        .count()
    )

    null_count = (
        df_clean
        .filter(F.col(column_name).isNull())
        .count()
    )

    observed_values = [
        row[column_name]
        for row in (
            df_clean
            .select(column_name)
            .distinct()
            .orderBy(column_name)
            .collect()
        )
    ]

    domain_validation_rows.append(
        (
            column_name,
            ", ".join(map(str, valid_values)),
            str(observed_values),
            invalid_count,
            null_count,
        )
    )

domain_validation_df = spark.createDataFrame(
    domain_validation_rows,
    schema="""
        COLUMN_NAME string,
        EXPECTED_VALUES string,
        OBSERVED_VALUES string,
        INVALID_RECORDS long,
        NULL_RECORDS long
    """,
)

display(domain_validation_df)

### Validation Interpretation

The domain-validation results compare the observed values in each selected variable with its approved range.

- `INVALID_RECORDS = 0` indicates that all non-null values comply with the expected domain.
- `NULL_RECORDS` are reported separately because some null values may be structurally valid.
- Null values in `ARR_DEL15` are expected for cancelled and diverted flights and are not automatically considered data errors.

Any non-zero invalid-record count must be investigated before business-rule validation is performed.

## 12. Business Rule Validation

Business-rule validation checks whether related variables are logically consistent with one another. These checks go beyond data types and permitted value ranges by validating the operational meaning of the records.

### 12.1 Departure Delay Indicator Consistency

The `DEP_DEL15` indicator should agree with the numerical value recorded in `DEP_DELAY`.

The following relationships are expected:

- If `DEP_DEL15 = 1`, then `DEP_DELAY` must be 15 minutes or greater.
- If `DEP_DEL15 = 0`, then `DEP_DELAY` must be less than 15 minutes.
- Records with null values are evaluated separately and are not counted as violations in this check.

No records are modified during this validation.

In [0]:
departure_delay_rule_summary = (
    df_clean
    .select(
        F.count("*").alias("TOTAL_ROWS"),
        F.sum(
            F.when(
                (F.col("DEP_DEL15") == 1)
                & F.col("DEP_DELAY").isNotNull()
                & (F.col("DEP_DELAY") < 15),
                1,
            ).otherwise(0)
        ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),
        F.sum(
            F.when(
                (F.col("DEP_DEL15") == 0)
                & F.col("DEP_DELAY").isNotNull()
                & (F.col("DEP_DELAY") >= 15),
                1,
            ).otherwise(0)
        ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),
        F.sum(
            F.when(
                F.col("DEP_DEL15").isNull()
                | F.col("DEP_DELAY").isNull(),
                1,
            ).otherwise(0)
        ).alias("ROWS_WITH_NULL_DEPARTURE_DELAY_FIELDS"),
    )
)

display(departure_delay_rule_summary)

### Validation Summary

The departure delay business rule validation confirms that the binary departure delay indicator (`DEP_DEL15`) is fully consistent with the recorded departure delay (`DEP_DELAY`).

No logical inconsistencies were identified:

- No records were found where `DEP_DEL15 = 1` while `DEP_DELAY < 15`.
- No records were found where `DEP_DEL15 = 0` while `DEP_DELAY ≥ 15`.

The remaining records with null departure delay values are expected operational cases and will be investigated separately before any cleaning decisions are applied.

### 12.2 Arrival Delay Indicator Consistency

The `ARR_DEL15` indicator should be logically consistent with the recorded arrival delay (`ARR_DELAY`).

The following rules are expected:

- If `ARR_DEL15 = 1`, then `ARR_DELAY` must be **15 minutes or greater**.
- If `ARR_DEL15 = 0`, then `ARR_DELAY` must be **less than 15 minutes**.
- Records with null arrival values are evaluated separately because they may correspond to cancelled or diverted flights.

In [0]:
arrival_delay_rule_summary = (
    df_clean
    .select(
        F.count("*").alias("TOTAL_ROWS"),

        F.sum(
            F.when(
                (F.col("ARR_DEL15") == 1)
                & F.col("ARR_DELAY").isNotNull()
                & (F.col("ARR_DELAY") < 15),
                1,
            ).otherwise(0)
        ).alias("INDICATOR_1_BUT_DELAY_BELOW_15"),

        F.sum(
            F.when(
                (F.col("ARR_DEL15") == 0)
                & F.col("ARR_DELAY").isNotNull()
                & (F.col("ARR_DELAY") >= 15),
                1,
            ).otherwise(0)
        ).alias("INDICATOR_0_BUT_DELAY_AT_LEAST_15"),

        F.sum(
            F.when(
                F.col("ARR_DEL15").isNull()
                | F.col("ARR_DELAY").isNull(),
                1,
            ).otherwise(0)
        ).alias("ROWS_WITH_NULL_ARRIVAL_DELAY_FIELDS"),
    )
)

display(arrival_delay_rule_summary)

### Validation Summary

The arrival delay business rule validation confirms that the binary arrival delay indicator (`ARR_DEL15`) is fully consistent with the recorded arrival delay (`ARR_DELAY`).

No logical inconsistencies were identified:

- No records were found where `ARR_DEL15 = 1` while `ARR_DELAY < 15`.
- No records were found where `ARR_DEL15 = 0` while `ARR_DELAY ≥ 15`.

The remaining records containing null arrival delay fields correspond to legitimate operational situations, such as cancelled or diverted flights, and therefore do not represent data quality issues.

### 12.3 Cancellation Code Consistency

Flights marked as cancelled should have an associated cancellation reason recorded in `CANCELLATION_CODE`.

The following business rules are validated:

- If `CANCELLED = 1`, then `CANCELLATION_CODE` should not be null.
- If `CANCELLED = 0`, then `CANCELLATION_CODE` is expected to be null.

These checks verify that cancellation information has been recorded consistently throughout the dataset.

In [0]:
cancellation_rule_summary = (
    df_clean
    .select(
        F.count("*").alias("TOTAL_ROWS"),

        F.sum(
            F.when(
                (F.col("CANCELLED") == 1)
                &
                (F.col("CANCELLATION_CODE").isNull()),
                1,
            ).otherwise(0)
        ).alias("CANCELLED_WITHOUT_REASON"),

        F.sum(
            F.when(
                (F.col("CANCELLED") == 0)
                &
                (F.col("CANCELLATION_CODE").isNotNull()),
                1,
            ).otherwise(0)
        ).alias("NOT_CANCELLED_WITH_REASON"),
    )
)

display(cancellation_rule_summary)

### Validation Summary

The cancellation business rule validation confirms that cancellation information is consistently recorded throughout the dataset.

No inconsistencies were identified:

- Every cancelled flight contains a valid cancellation reason.
- No non-cancelled flight contains an unnecessary cancellation code.

These results indicate that the cancellation-related variables comply with the expected operational business rules defined by the BTS dataset.

### 12.4 Diversion Consistency

Flights marked as diverted follow a different operational process from normally completed flights. Therefore, the arrival delay indicator (`ARR_DEL15`) is expected to be unavailable for diverted flights.

The following business rules are validated:

- If `DIVERTED = 1`, then `ARR_DEL15` should be null.
- If `DIVERTED = 0`, then `ARR_DEL15` should normally contain a valid value unless another documented operational condition applies.

These checks verify that diversion information is recorded consistently within the dataset.

In [0]:
diversion_rule_summary = (
    df_clean
    .select(
        F.count("*").alias("TOTAL_ROWS"),

        F.sum(
            F.when(
                (F.col("DIVERTED") == 1)
                &
                (F.col("ARR_DEL15").isNotNull()),
                1,
            ).otherwise(0)
        ).alias("DIVERTED_WITH_TARGET"),

        F.sum(
    F.when(
        (F.col("DIVERTED") == 0)
        &
        (F.col("CANCELLED") == 0)
        &
        (F.col("ARR_DEL15").isNull()),
        1,
    ).otherwise(0)
).alias("NON_DIVERTED_NON_CANCELLED_WITH_NULL_TARGET")
    )
)

display(diversion_rule_summary)

### Validation Summary

The diversion consistency check confirms that diversion and arrival-target values are recorded correctly.

No inconsistencies were identified:

- No diverted flights contain a non-null `ARR_DEL15` value.
- No non-diverted, non-cancelled flights contain a null `ARR_DEL15` value.

This result confirms that the arrival delay target is structurally consistent with both cancellation and diversion status.

### 12.5 Delay-Cause Variable Consistency

The BTS delay-cause variables describe the number of delay minutes attributed to specific causes:

- `CARRIER_DELAY`
- `WEATHER_DELAY`
- `NAS_DELAY`
- `SECURITY_DELAY`
- `LATE_AIRCRAFT_DELAY`

These fields are generally expected to be populated only for flights with a reportable arrival delay. This validation checks whether non-delayed flights contain positive delay-cause values.

No records are modified during this validation.

## 13. Cleaning Summary

The data cleaning process has been completed following a series of validation and quality assessment procedures.

The completed activities include:

- Loading the managed Delta table.
- Validating the approved dataset schema.
- Standardizing data types.
- Standardizing categorical text fields.
- Verifying successful date conversion.
- Detecting duplicate records.
- Assessing missing values.
- Investigating missing target values.
- Validating domain constraints.
- Validating business rules.

The cleaning process identified one incomplete record containing a missing target value (`ARR_DEL15`) despite the flight not being cancelled or diverted. This record was removed because it cannot be used for supervised machine learning.

All remaining missing values were determined to represent legitimate operational conditions rather than data quality issues and were therefore retained.

## 14. Save Clean Dataset

The cleaned dataset is stored in Delta format within the processed layer of the project data lake. This dataset serves as the input to the Feature Engineering notebook.

Saving the cleaned dataset separately preserves the original raw dataset and supports reproducibility throughout the analytical pipeline.

In [0]:
# Save the cleaned dataset to the processed layer

(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .save(CLEAN_DELTA_PATH)
)

print("Clean dataset saved successfully.")
print(f"Location: {CLEAN_DELTA_PATH}")
print(f"Total cleaned records: {df_clean.count():,}")

### Validation Summary

The cleaned dataset is reloaded from the processed layer to verify that it was successfully written and remains readable. This verification confirms the integrity of the saved Delta dataset before it is used in the Feature Engineering stage.

In [0]:
df_processed = spark.read.format("delta").load(CLEAN_DELTA_PATH)

print(f"Processed records: {df_processed.count():,}")

display(df_processed.limit(10))

## 15. Save the Cleaned Dataset as a Managed Delta Table

The cleaned dataset is saved as a managed Delta table in the
`workspace.default` schema. Registering the dataset as a Unity Catalog table
allows it to be queried through SQL, accessed by downstream notebooks, governed
through Unity Catalog permissions, and displayed in Catalog Explorer.

The original `flights_raw` table remains unchanged.

In [0]:
CLEAN_TABLE = "workspace.default.flights_clean"

(
    df_clean.writeTo(CLEAN_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Cleaned managed Delta table created successfully.")
print(f"Table: {CLEAN_TABLE}")
print(f"Total records: {spark.table(CLEAN_TABLE).count():,}")